# WaferDefect Studio — Cloud-GPU Training

**Diffusion augmentation + Lightweight ViT + Grad-CAM — WM-811K → MixedWM38**

Runtime → Change runtime type → **GPU** (T4 is enough)

Pipeline: download data (kagglehub) → train diffusion → train 3 classifier arms → evaluate + export

In [ ]:
# 1 · Clone / upload the project & install deps
# Easiest: zip the project locally and upload, or mount Drive:
#   from google.colab import drive; drive.mount('/content/drive')
#   %cd /content/drive/MyDrive/Wafer-MAP

%cd /content
!git clone https://github.com/<YOU>/Wafer-MAP.git Wafer-MAP 2>/dev/null || echo 'using uploaded copy'
%cd Wafer-MAP 2>/dev/null || %cd /content
!pip install -q kagglehub scipy streamlit
!pip install -q -r requirements.txt

In [ ]:
# 2 · Download datasets via kagglehub (needs a free Kaggle token on Colab:
# kagglehub reads ~/.kaggle/kaggle.json or KAGGLE_USERNAME/KAGGLE_KEY env vars)
import kagglehub, shutil, os

os.makedirs('data', exist_ok=True)

p = kagglehub.dataset_download('co1d7era/mixedtype-wafer-defect-datasets')
print('MixedWM38 ->', p)
if not os.path.exists('data/MixedWM38'):
    shutil.copytree(p, 'data/MixedWM38', dirs_exist_ok=True)

p2 = kagglehub.dataset_download('emphymachine/sample-wafermap-data')
print('Sample ->', p2)
if not os.path.exists('data/sample_wafermap'):
    shutil.copytree(p2, 'data/sample_wafermap', dirs_exist_ok=True)

# WM-811K LSWMD.pkl (~2 GB): download once and keep on Drive, or:
# p3 = kagglehub.dataset_download('qingyi/wm811k-wafer-map')
!ls -la data/

In [ ]:
# 3 · Train the latent-diffusion augmentation model (~30-60 min on T4)
!python src/train_diffusion.py

In [ ]:
# 4 · Train the three classifier arms (~1-3 h total on T4)
!python src/train.py --arms cnn vit vit_diff

In [ ]:
# 5 · Evaluate: ablation + cross-dataset robustness + explainability grids
!python src/evaluate.py

In [ ]:
# 6 · Inspect results
import pandas as pd
from IPython.display import Image, display
display(Image('outputs/explain/grid_vit_diff.png'))
pd.read_csv('outputs/cross_dataset_results.csv')

In [ ]:
# 7 · Zip artifacts for download back to your machine
!zip -qr wafer_results.zip outputs app.py src
from google.colab import files
files.download('wafer_results.zip')